In [1]:
import sys
import pandas as pd
import numpy as np


sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

from estimation import estimate_var, estimate_arima, cast_to_base_unit
from plots import plot_prediction
from utils import _convert_to_datetime

In [2]:
import pandas as pd

In [3]:
series_name = "PCEC96"
reference_date = "2024-08-01"
n_periods = 72

ds = pd.read_parquet("../data/04_feature/selected_series.parquet")
ds_base = pd.read_parquet("../data/02_intermediate/non_transformed_data.parquet")

ds_spec = pd.read_csv("../data/02_intermediate/variable.csv")


In [7]:
var_model_result = estimate_var(
    ds=ds,
    ds_base=ds_base,
    spec=ds_spec,
    ref_date_col="ReferenceDate",
    series_name=series_name,
    reference_date=reference_date,
    n_periods=n_periods,
)

Time = var_model_result['predictions']['backcast'].index
plot_prediction(dt=Time, y_pred=var_model_result['predictions']['backcast'], y_actual=var_model_result['actual'].loc[Time], mode="lines+markers")

In [5]:
Rhat, R, Time, cutoff_date = cast_to_base_unit(ds_base, var_model_result, ds_spec, series_name)
header = [series_name]
Rhat_df = pd.DataFrame(Rhat, columns=header, index=Time)
R_df = pd.DataFrame(R, columns=header, index=Time)

Z_df = pd.DataFrame(ds[series_name], columns=header, index=Time)

dt = Time
plot_prediction(dt=dt, y_pred=Rhat_df.loc[dt][series_name], y_actual=R_df.loc[dt][series_name], mode="lines+markers")


In [6]:
ar_model_result = estimate_arima(
    ds=ds,
    ds_base=ds_base,
    spec=ds_spec,
    ref_date_col="ReferenceDate",
    series_name=series_name,
    reference_date=reference_date,
    n_periods=n_periods,
)

Time = ar_model_result['predictions']['backcast'].index
plot_prediction(dt=Time, y_pred=ar_model_result['predictions']['backcast'], y_actual=ar_model_result['actual'].loc[Time], mode="lines+markers")

In [5]:
Shat, S, Time, cutoff_date = cast_to_base_unit(ds_base, ar_model_result, ds_spec, series_name)
header = [series_name]
Shat_df = pd.DataFrame(Shat, columns=header, index=Time)
S_df = pd.DataFrame(S, columns=header, index=Time)

Z_df = pd.DataFrame(ds[series_name], columns=header, index=Time)

dt = Time
plot_prediction(dt=dt, y_pred=Shat_df.loc[dt][series_name], y_actual=S_df.loc[dt][series_name], mode="lines+markers")
